# Klasifikasi Kualitas Udara DKI Jakarta 2023
**Metode: k-NN + SMOTETomek | Data: ISPU DKI Jakarta 2023**

In [ ]:
!pip install imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)
from imblearn.combine import SMOTETomek

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('[OK] Semua library berhasil diimpor.')

## 1. Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

FILE_PATH = '/content/drive/MyDrive/Data Mining Project/data-indeks-standar-pencemar-udara-(ispu)-di-provinsi-dki-jakarta-2023-komponen-data.csv'

try:
    df_raw = pd.read_csv(FILE_PATH)
    print(f'[SUKSES] Dataset dimuat: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom')
except FileNotFoundError:
    import os
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if csv_files:
        FILE_PATH = csv_files[0]
        df_raw = pd.read_csv(FILE_PATH)
        print(f'[SUKSES] Menggunakan: {FILE_PATH}')
    else:
        raise FileNotFoundError('Upload file CSV ISPU DKI 2023 terlebih dahulu.')

df_raw.columns = df_raw.columns.str.lower()
print('\n--- Struktur Dataset ---')
print(df_raw.info())
print('\n--- 5 Baris Pertama ---')
df_raw.head()

## 2. Preprocessing & Pembersihan Data

**Perbaikan yang diterapkan:**
- Menangani semua varian teks missing (`TIDAK ADA DATA`, `-`, `''`, dll)
- Logika label diperbaiki: hanya butuh PM10 **atau** PM2.5 yang ada (bukan keduanya)
- Imputasi median dilakukan **setelah** split → mencegah data leakage

In [ ]:
print('[INFO] Preprocessing data...')

df_clean = df_raw.copy()
features_numerik = ['pm_sepuluh', 'pm_duakomalima', 'sulfur_dioksida',
                    'karbon_monoksida', 'ozon', 'nitrogen_dioksida']

# FIX 1: Tangani semua varian teks missing
MISSING_VALS = {'TIDAK ADA DATA', '-', 'NAN', 'NONE', '', 'N/A', 'NA'}
for col in features_numerik:
    if col in df_clean.columns:
        df_clean[col] = (df_clean[col]
                         .astype(str).str.strip().str.upper()
                         .replace({v: np.nan for v in MISSING_VALS}))
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print('\n--- Missing Values Per Kolom ---')
print(df_clean[features_numerik].isnull().sum())

# FIX 2: Perbaiki logika label
# Asli: jika PM2.5 ATAU PM10 kosong → Tidak Ada Data  (terlalu ketat)
# Baru: jika PM2.5 DAN PM10 keduanya kosong → Tidak Ada Data
def tentukan_kategori(row):
    if pd.isna(row['pm_duakomalima']) and pd.isna(row['pm_sepuluh']):
        return 'Tidak Ada Data'
    checks = [
        ('pm_sepuluh',        150),
        ('pm_duakomalima',    55.4),
        ('sulfur_dioksida',   180),
        ('karbon_monoksida',  8000),
        ('ozon',              235),
        ('nitrogen_dioksida', 200),
    ]
    for col, thr in checks:
        val = row.get(col)
        if pd.notna(val) and val > thr:
            return 'Tidak Sehat'
    return 'Sehat'

df_clean['kategori_baru'] = df_clean.apply(tentukan_kategori, axis=1)

print('\n--- Distribusi Kategori (Total Data) ---')
print(df_clean['kategori_baru'].value_counts())
print(f'\nTotal: {len(df_clean)} baris')

## 3. Split & Scaling (Cegah Data Leakage)

In [ ]:
df_missing = df_clean[df_clean['kategori_baru'] == 'Tidak Ada Data'].copy()
df_model   = df_clean[df_clean['kategori_baru'] != 'Tidak Ada Data'].copy()

print(f'Data untuk modeling  : {len(df_model)} baris')
print(f'Data Tidak Ada Data  : {len(df_missing)} baris (dilewati)')

X = df_model[features_numerik]
y = df_model['kategori_baru']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# FIX 3: Imputasi median HANYA dari training set
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test  = X_test.fillna(train_medians)  # gunakan median train, bukan test!

print(f'\nTrain: {X_train.shape[0]} baris | Test: {X_test.shape[0]} baris')
print('\nDistribusi kelas train:')
print(y_train.value_counts())

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('[OK] Scaling selesai.')

## 4. SMOTETomek — Penanganan Imbalanced Data

In [ ]:
print('[INFO] Menerapkan SMOTETomek...')
print('\nDistribusi SEBELUM SMOTETomek:')
print(pd.Series(y_train).value_counts())

smotetomek = SMOTETomek(random_state=42)
X_train_resampled, y_train_resampled = smotetomek.fit_resample(X_train_scaled, y_train)

print('\nDistribusi SETELAH SMOTETomek:')
print(pd.Series(y_train_resampled).value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_map = {'Tidak Sehat': '#e74c3c', 'Sehat': '#2ecc71'}

for ax, data, title in zip(
    axes,
    [pd.Series(y_train).value_counts(), pd.Series(y_train_resampled).value_counts()],
    ['Sebelum SMOTETomek', 'Setelah SMOTETomek']
):
    cols = [colors_map.get(l, '#95a5a6') for l in data.index]
    ax.bar(data.index, data.values, color=cols, edgecolor='white', width=0.5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Jumlah Data', fontsize=11)
    ax.set_xlabel('Kategori', fontsize=11)
    for i, (label, val) in enumerate(data.items()):
        ax.text(i, val + 5, str(val), ha='center', fontsize=12, fontweight='bold')

plt.suptitle('Gambar 0. Distribusi Kelas Sebelum & Sesudah SMOTETomek', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('distribusi_smotetomek.png', dpi=300, bbox_inches='tight')
plt.show()
print('[SUKSES] Grafik disimpan.')

## 5. Hyperparameter Tuning — Cari k Optimal

**FIX 4:** Gunakan `f1_macro` sebagai scoring (lebih tepat untuk data yang tidak seimbang)

In [ ]:
print('[INFO] GridSearchCV mencari k terbaik dengan F1-macro...')

param_grid = {'n_neighbors': [3, 5, 7, 9, 11, 13, 15]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=cv,
    scoring='f1_macro',   # lebih adil untuk imbalanced
    n_jobs=-1
)
grid_search.fit(X_train_resampled, y_train_resampled)

k_terbaik = grid_search.best_params_['n_neighbors']
skor_cv   = grid_search.best_score_

print(f'\nk terbaik   : {k_terbaik}')
print(f'F1-macro CV : {skor_cv * 100:.2f}%')

print('\n--- Perbandingan Semua Nilai k ---')
res = grid_search.cv_results_
for k, mean, std in zip(res['param_n_neighbors'],
                         res['mean_test_score'],
                         res['std_test_score']):
    print(f'  k={k:>2} | F1-macro: {mean*100:.2f}% +/- {std*100:.2f}%')

# Grafik
plt.figure(figsize=(9, 4))
k_vals  = [p['n_neighbors'] for p in res['params']]
means   = res['mean_test_score']
stds    = res['std_test_score']
plt.plot(k_vals, means * 100, 'o-', color='#2980b9', lw=2, markersize=7, label='F1-macro CV')
plt.fill_between(k_vals, (means-stds)*100, (means+stds)*100, alpha=0.15, color='#2980b9')
plt.axvline(x=k_terbaik, color='#e74c3c', linestyle='--', label=f'k optimal = {k_terbaik}')
plt.title('Gambar 3. F1-macro CV per Nilai k', fontsize=12, pad=15)
plt.xlabel('Nilai k', fontsize=11)
plt.ylabel('F1-macro (%)', fontsize=11)
plt.xticks(k_vals)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('grafik_k_vs_f1.png', dpi=300)
plt.show()

## 6. Modeling Akhir dengan k Optimal

In [ ]:
knn_final = KNeighborsClassifier(n_neighbors=k_terbaik)
knn_final.fit(X_train_resampled, y_train_resampled)
y_pred       = knn_final.predict(X_test_scaled)
y_pred_proba = knn_final.predict_proba(X_test_scaled)

akurasi_final = accuracy_score(y_test, y_pred)

# FIX 5: Tambahkan ROC-AUC
le = LabelEncoder()
y_test_enc = le.fit_transform(y_test)
idx_ts = list(knn_final.classes_).index('Tidak Sehat')
roc_auc = roc_auc_score(y_test_enc, y_pred_proba[:, idx_ts])

print('\n' + '='*55)
print('   HASIL EVALUASI AKHIR MODEL k-NN (K OPTIMAL)')
print('='*55)
print(f'  k yang digunakan    : {k_terbaik}')
print(f'  Akurasi Test Set    : {akurasi_final * 100:.2f}%')
print(f'  F1-macro CV         : {skor_cv * 100:.2f}%')
print(f'  ROC-AUC             : {roc_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

## 7. Visualisasi Hasil

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=['Sehat', 'Tidak Sehat'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Sehat', 'Tidak Sehat'],
            yticklabels=['Sehat', 'Tidak Sehat'], ax=axes[0])
axes[0].set_title(f'Gambar 1. Confusion Matrix k-NN (k={k_terbaik})', fontsize=12, pad=12)
axes[0].set_xlabel('Prediksi Model', fontsize=10)
axes[0].set_ylabel('Kondisi Aktual', fontsize=10)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test_enc, y_pred_proba[:, idx_ts])
axes[1].plot(fpr, tpr, color='#2980b9', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_title('Gambar 2. ROC Curve (Kelas: Tidak Sehat)', fontsize=12, pad=12)
axes[1].set_xlabel('False Positive Rate', fontsize=10)
axes[1].set_ylabel('True Positive Rate', fontsize=10)
axes[1].legend(fontsize=10)

# Pie Chart
proporsi = df_clean['kategori_baru'].value_counts()
cmap_pie = {'Tidak Sehat': '#e74c3c', 'Sehat': '#2ecc71', 'Tidak Ada Data': '#f1c40f'}
axes[2].pie(proporsi, labels=proporsi.index, autopct='%1.1f%%',
            startangle=140,
            colors=[cmap_pie.get(l,'#95a5a6') for l in proporsi.index])
axes[2].set_title('Gambar 3. Proporsi Kualitas Udara DKI 2023', fontsize=12, pad=12)

plt.tight_layout()
plt.savefig('visualisasi_hasil_final.png', dpi=300, bbox_inches='tight')
plt.show()
print('[SUKSES] Visualisasi gabungan disimpan.')

## 8. Simpan Model

In [ ]:
import pickle, os

with open('knn_model.pkl', 'wb') as f: pickle.dump(knn_final, f)
with open('scaler.pkl', 'wb') as f:    pickle.dump(scaler, f)
with open('train_medians.pkl', 'wb') as f:  # FIX 6: simpan median imputasi
    pickle.dump(train_medians.to_dict(), f)

metadata = {
    'k_terbaik'      : int(k_terbaik),
    'akurasi_test'   : float(round(akurasi_final, 4)),
    'f1_macro_cv'    : float(round(skor_cv, 4)),
    'roc_auc'        : float(round(roc_auc, 4)),
    'features'       : features_numerik,
    'label_classes'  : list(knn_final.classes_),
    'train_size'     : int(X_train.shape[0]),
    'test_size'      : int(X_test.shape[0]),
    'resampled_train': int(X_train_resampled.shape[0]),
    'threshold_pm10' : 150,
    'threshold_pm25' : 55.4,
    'threshold_so2'  : 180,
    'threshold_co'   : 8000,
    'threshold_o3'   : 235,
    'threshold_no2'  : 200,
    'dataset'        : 'ISPU DKI Jakarta 2023',
}
with open('model_metadata.pkl', 'wb') as f: pickle.dump(metadata, f)

print('Metadata Model:')
for k, v in metadata.items():
    print(f'  {k:<22}: {v}')

print('\nFile tersimpan:')
for fn in ['knn_model.pkl','scaler.pkl','train_medians.pkl','model_metadata.pkl']:
    print(f'  {fn} ({os.path.getsize(fn):,} bytes)')

## 9. Generate app.py & requirements.txt

In [ ]:
req = (
    'streamlit>=1.35.0\n'
    'scikit-learn>=1.4.0\n'
    'pandas>=2.0.0\n'
    'numpy>=1.26.0\n'
    'matplotlib>=3.8.0\n'
    'seaborn>=0.13.0\n'
    'imbalanced-learn>=0.12.0\n'
)
with open('requirements.txt', 'w') as f: f.write(req)
print('[OK] requirements.txt')

app_lines = [
    'import streamlit as st',
    'import pickle, numpy as np, pandas as pd',
    '',
    'st.set_page_config(page_title="Kualitas Udara DKI", page_icon="\U0001f32b", layout="wide")',
    '',
    '@st.cache_resource',
    'def load_artifacts():',
    '    with open("knn_model.pkl","rb") as f: model = pickle.load(f)',
    '    with open("scaler.pkl","rb") as f: scaler = pickle.load(f)',
    '    with open("train_medians.pkl","rb") as f: medians = pickle.load(f)',
    '    with open("model_metadata.pkl","rb") as f: meta = pickle.load(f)',
    '    return model, scaler, medians, meta',
    '',
    'model, scaler, medians, meta = load_artifacts()',
    '',
    'st.title("Klasifikasi Kualitas Udara DKI Jakarta")',
    'st.markdown("Berbasis **K-NN + SMOTETomek** | Data ISPU 2023")',
    'st.divider()',
    '',
    'with st.sidebar:',
    '    st.header("Info Model")',
    '    st.metric("k Optimal", meta["k_terbaik"])',
    '    st.metric("Akurasi Test", f"{meta[\"akurasi_test\"]*100:.2f}%")',
    '    st.metric("F1-macro CV", f"{meta[\"f1_macro_cv\"]*100:.2f}%")',
    '    st.metric("ROC-AUC", f"{meta[\"roc_auc\"]:.4f}")',
    '',
    'col1, col2 = st.columns(2)',
    'with col1:',
    '    st.subheader("Input Nilai ISPU")',
    '    pm10 = st.number_input("PM10 (ug/m3)",  0.0, 600.0, float(medians.get("pm_sepuluh",50)))',
    '    pm25 = st.number_input("PM2.5 (ug/m3)", 0.0, 500.0, float(medians.get("pm_duakomalima",30)))',
    '    so2  = st.number_input("SO2 (ug/m3)",   0.0, 1000.0, float(medians.get("sulfur_dioksida",20)))',
    '    co   = st.number_input("CO (ug/m3)",    0.0, 50000.0, float(medians.get("karbon_monoksida",2000)), step=100.0)',
    '    o3   = st.number_input("O3 (ug/m3)",   0.0, 800.0, float(medians.get("ozon",30)))',
    '    no2  = st.number_input("NO2 (ug/m3)",  0.0, 1000.0, float(medians.get("nitrogen_dioksida",25)))',
    '    btn  = st.button("Prediksi", type="primary", use_container_width=True)',
    'with col2:',
    '    st.subheader("Hasil Prediksi")',
    '    if btn:',
    '        inp = np.array([[pm10,pm25,so2,co,o3,no2]])',
    '        pred  = model.predict(scaler.transform(inp))[0]',
    '        proba = model.predict_proba(scaler.transform(inp))[0]',
    '        if pred=="Sehat":',
    '            st.success("SEHAT - Aman untuk aktivitas luar ruangan.")',
    '        else:',
    '            st.error("TIDAK SEHAT - Kurangi aktivitas luar ruangan.")',
    '        for label,p in zip(model.classes_,proba):',
    '            st.progress(float(p), text=f"{label}: {p*100:.1f}%")',
    '    else:',
    '        st.info("Isi nilai di kiri lalu klik Prediksi.")',
]
with open('app.py','w') as f:
    f.write('\n'.join(app_lines))
print('[OK] app.py dibuat')
print('\nFile siap untuk deploy Streamlit Community Cloud:')
print('  app.py, requirements.txt, knn_model.pkl, scaler.pkl, train_medians.pkl, model_metadata.pkl')